In [42]:
import os
import pandas as pd
from pathlib import Path
import gc

# Run garbage collection
gc.collect()

data_folder = Path("data/train.csv")
data_path = os.path.join(os.getcwd(), data_folder)

print(data_path)

# # submission data format
# df = pd.read_parquet(data_path)
# df.head()

df = pd.read_csv(data_path)
df.head()

c:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\train.csv


,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",qwen3.8-max
1,0041ac86ee93,p073,Data science communities matter because they a...,claude-sonnet-5
2,005a19083eae,p120,"In the final twilight of time, the last embers...",gemini-3.6-flash
3,009319596f10,p073,Data science communities matter because they t...,gpt-5.6-terra
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",glm-5.3-flash


In [44]:
# Applying label encoder
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["model"] = le.fit_transform(df["model"])
df.head(5)

,id,prompt_id,text,model
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",17
1,0041ac86ee93,p073,Data science communities matter because they a...,4
2,005a19083eae,p120,"In the final twilight of time, the last embers...",6
3,009319596f10,p073,Data science communities matter because they t...,14
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",10


### EDA

In [47]:
import nltk
nltk.download("punkt")
from nltk.tokenize import word_tokenize, sent_tokenize

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ps302\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ps302\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
df["num_chars"] = df["text"].apply(len)
df["num_words"] = df["text"].apply(lambda x: len(word_tokenize(x)))
df["num_sents"] = df["text"].apply(lambda x: len(sent_tokenize(x)))
df.head()

,id,prompt_id,text,model,num_chars
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",17,1143
1,0041ac86ee93,p073,Data science communities matter because they a...,4,518
2,005a19083eae,p120,"In the final twilight of time, the last embers...",6,474
3,009319596f10,p073,Data science communities matter because they t...,14,1724
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",10,541


### Model Training

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import numpy as np
# cv = CountVectorizer()
# tfidf = TfidfVectorizer(max_features=3000)
cv = TfidfVectorizer(max_features=3000)
char_tfidf = TfidfVectorizer(analyzer='char', ngram_range=(2, 5), max_features=5000)
word_tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)


In [52]:
df.head()

,id,prompt_id,text,model,num_chars,num_words,num_sents
0,0018b3765972,p081,"When a language dies, a particular astronomy d...",17,1143,222,8
1,0041ac86ee93,p073,Data science communities matter because they a...,4,518,74,4
2,005a19083eae,p120,"In the final twilight of time, the last embers...",6,474,89,4
3,009319596f10,p073,Data science communities matter because they t...,14,1724,263,9
4,00d3e8c92596,p097,"Deep inside each ear, three tiny loops of bone...",10,541,114,4


In [54]:
from sklearn.model_selection import train_test_split
df_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
print(df_train.shape, df_test.shape)

(2159, 7) (540, 7)


In [55]:
y_train = df_train["model"].values
y_test = df_test["model"].values

print(y_train.shape, y_test.shape)

(2159,) (540,)


In [ ]:
X_train_text = cv.fit_transform(df_train["text"]).toarray()
X_train_num = df_train[["num_chars", "num_words", "num_sents"]].values


X_test_text = cv.transform(df_test["text"]).toarray()
X_test_num = df_test[["num_chars", "num_words", "num_sents"]].values


(2159, 3003)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(X_train_num)
X_test_num_scaled = scaler.transform(X_test_num)


In [ ]:
X_train = np.hstack((X_train_text, X_train_num_scaled))
X_test = np.hstack((X_test_text, X_test_num_scaled))
print(X_train.shape,X_test.shape)

(540, 3003)

In [58]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

# Initialize and fit XGBoost
xgb = XGBClassifier(
    objective="multi:softprob",      # Multiclass classification
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42,
    tree_method="hist",              # Fast histogram algorithm
    n_jobs=-1
)

xgb.fit(X_train, y_train)
y_pred = xgb.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


Accuracy Score: 0.1685
Recall Score: 0.1773


In [ ]:
from sklearn.linear_model import LogisticRegression
clf = LogisticRegression(max_iter=1000, C=1.0)

clf.fit(X_train, y_train)
y_pred = clf.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")



In [60]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

gnb = GaussianNB()
gnb.fit(X_train, y_train)
y_pred = gnb.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred,  average='macro'):.4f}")
# print(f"Confusion Metrics: {confusion_matrix(y_test, y_pred)}")

Accuracy Score: 0.0259
Recall Score: 0.0275


In [62]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

mnb = MultinomialNB()
mnb.fit(X_train, y_train)
y_pred = mnb.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred,  average='macro'):.4f}")
# print(f"Confusion Metrics: {confusion_matrix(y_test, y_pred)}")

Accuracy Score: 0.0685
Recall Score: 0.0859


In [64]:
from sklearn.naive_bayes import GaussianNB, MultinomialNB, BernoulliNB
from sklearn.metrics import accuracy_score, confusion_matrix, recall_score

bnb = BernoulliNB()
bnb.fit(X_train, y_train)
y_pred = bnb.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred,  average='macro'):.4f}")
# print(f"Confusion Metrics: \n {confusion_matrix(y_test, y_pred)}")

Accuracy Score: 0.0704
Recall Score: 0.0804


In [ ]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

# 'linear' kernel is usually best and fastest for text/NLP features (TF-IDF, Bag of Words)
# For non-linear, use kernel='rbf' (default)
svm = SVC(kernel="linear", random_state=42)
svm.fit(X_train, y_train)
y_pred = svm.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


In [ ]:
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix

lgb = LGBMClassifier(
    objective="multiclass",
    n_estimators=100,
    learning_rate=0.08,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbose=-1  # Suppress training warnings/info
)

lgb.fit(X_train, y_train)
y_pred = lgb.predict(X_test)

print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}")
print(f"Recall Score: {recall_score(y_test, y_pred, average='macro'):.4f}")
# print(f"Confusion Matrix:\n{confusion_matrix(y_test, y_pred)}")


In [ ]:
from sklearn.pipeline import FeatureUnion
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, recall_score
import numpy as np

# 1. Feature extractors
word_vec = TfidfVectorizer(ngram_range=(1, 2), max_features=40000, sublinear_tf=True)
char_vec = TfidfVectorizer(analyzer='char', ngram_range=(3, 5), max_features=40000, sublinear_tf=True)

# Fit on text
X_train_w = word_vec.fit_transform(df_train["text"]).toarray()
X_test_w = word_vec.transform(df_test["text"]).toarray()

X_train_c = char_vec.fit_transform(df_train["text"]).toarray()
X_test_c = char_vec.transform(df_test["text"]).toarray()

# Scale lengths
scaler = StandardScaler()
X_train_num_scaled = scaler.fit_transform(df_train[["num_chars", "num_words", "num_sents"]])
X_test_num_scaled = scaler.transform(df_test[["num_chars", "num_words", "num_sents"]])

# Combine all
X_train_combined = np.hstack((X_train_w, X_train_c, X_train_num_scaled))
X_test_combined = np.hstack((X_test_w, X_test_c, X_test_num_scaled))

# 2. Fit Logistic Regression
lr = LogisticRegression(max_iter=1000, C=1.0)
lr.fit(X_train_combined, y_train)

y_pred_lr = lr.predict(X_test_combined)
print(f"Logistic Regression Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")
print(f"Logistic Regression Recall: {recall_score(y_test, y_pred_lr, average='macro'):.4f}")


In [ ]:
# import pandas as pd
# df = pd.read_parquet(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\sample_submission.parquet")
# df.head()

In [ ]:
# test = pd.read_csv(r"C:\Users\ps302\OneDrive\Desktop\Name_that_LLM\data\test.csv")
# test.head()

In [ ]:
# len(test)